# Prepare Landsat Data
## GEOG 6160 Final Project
## Magnus Tveit

This notebook takes Landsat 8/9 Collection 2 Level 2 Data, specifically bands 2-6. 

- B2 - Blue
- B3 - Green
- B4 - Red
- B5 - NIR
- B6 - SWIR1

It does two things:

1. Clips all GeoTIFF files in `RawData/` to the Colorado and Lake Powell delta extent
2. Saves the outputs to `Data/`
3. Calculates NDVI (Normalized Difference Vegetation Index) and NDWI (Normalized Difference Water Index)
4. Save then as additional bands with the `data/` folder

Input/Output Structure
```
Data/ (or RawData/)
   2013/   LC8_...__SR_B2.TIF   ... _SR_NDVI.TIF   _SR_NDWI.TIF
   2014/   LC8_...__SR_B2.TIF   ... _SR_NDVI.TIF   _SR_NDWI.TIF
   2015/   ...
```

### Set Up

In [4]:
import os
import numpy as np
import rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer

### Clip Data

In [5]:
# Paths
base_dir = os.getcwd()
raw_dir  = os.path.join(base_dir, "RawData")
out_dir  = os.path.join(base_dir, "Data")

# Bounds in WGS84, reprojected to EPSG:32612
west, south, east, north = -110.50, 37.75, -110.38, 37.89
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32612", always_xy=True)
x_min, y_min = transformer.transform(west, south)
x_max, y_max = transformer.transform(east, north)
clip_bounds = (x_min, y_min, x_max, y_max)

# Process each year folder
for year in sorted(os.listdir(raw_dir)):
    year_in  = os.path.join(raw_dir, year)
    year_out = os.path.join(out_dir, year)
    os.makedirs(year_out, exist_ok=True)

    tif_files = sorted([f for f in os.listdir(year_in) if f.lower().endswith((".tif", ".tiff"))])

    for fname in tif_files:
        in_path  = os.path.join(year_in, fname)
        out_path = os.path.join(year_out, fname)

        with rasterio.open(in_path) as src:
            # Derive a rasterio window from the clip bounds
            window        = from_bounds(*clip_bounds, transform=src.transform)
            win_transform = src.window_transform(window)
            data          = src.read(window=window)

            # Update the raster profile for the clipped dimensions
            profile = src.profile.copy()
            profile.update({
                "height":    data.shape[1],
                "width":     data.shape[2],
                "transform": win_transform,
            })

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

### Add NDVI and NDWI bands

- **NDVI** = (B4 - B3) / (B4 + B3)
- **NDWI** = (B2 - B4) / (B2 + B4)

In [6]:
base_dir = os.getcwd()
data_dir = os.path.join(base_dir, "Data")

for year in sorted(os.listdir(data_dir)):
    year_dir = os.path.join(data_dir, year)
    tif_files = os.listdir(year_dir)

    # Locate the three required bands
    b3 = next(f for f in tif_files if f.endswith("_SR_B3.TIF"))
    b4 = next(f for f in tif_files if f.endswith("_SR_B4.TIF"))
    b5 = next(f for f in tif_files if f.endswith("_SR_B5.TIF"))

    with rasterio.open(os.path.join(year_dir, b5)) as src:
        NIR = src.read(1).astype("float32")
        profile = src.profile.copy()

    with rasterio.open(os.path.join(year_dir, b4)) as src:
        RED = src.read(1).astype("float32")

    with rasterio.open(os.path.join(year_dir, b3)) as src:
        GREEN = src.read(1).astype("float32")

    profile.update(dtype="float32", count=1)

    # NDVI
    ndvi = (NIR - RED) / (NIR + RED)
    ndvi_path = os.path.join(year_dir, b5.replace("_SR_B5.TIF", "_SR_NDVI.TIF"))
    with rasterio.open(ndvi_path, "w", **profile) as dst:
        dst.write(ndvi, 1)

    # NDWI
    ndwi = (GREEN - NIR) / (GREEN + NIR)
    ndwi_path = os.path.join(year_dir, b5.replace("_SR_B5.TIF", "_SR_NDWI.TIF"))
    with rasterio.open(ndwi_path, "w", **profile) as dst:
        dst.write(ndwi, 1)